# 60 - Benchmark: RAF-DB Dataset (Self Train-Test)

**Dataset:** RAF-DB — 14,449 gambar basic-emotion (11,565 train / 2,884 test) dari Kaggle shuvoalok.
**Evaluasi:** Official train/test split (dari paper original).
**Model:** Semua model + Late Fusion (CNN, FCNN, Intermediate, CNN_TL, Intermediate_TL, Late_Fusion).
**Skenario:** B1 (Baseline) — tidak pakai class weights / augmentasi agar konsisten dengan benchmark standar.
**Metrik:** Macro F1, Micro F1 (=accuracy), Weighted F1 (semua di-print).

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'benchmark' / 'rafdb'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    ('CNN', EmotionCNN, 'cnn', 0.0001),
    ('FCNN', EmotionFCNN, 'fcnn', 0.0001),
    ('Intermediate', IntermediateFusion, 'fusion', 0.0001),
    ('CNN_TL', EmotionCNNTransfer, 'cnn', 0.00005),
    ('Intermediate_TL', IntermediateFusionTransfer, 'fusion', 0.00005),
]

print('Setup complete.')

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
# ── Helper Functions ──

def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == 'cnn': ds = TensorDataset(img_t, y_t)
    elif model_type == 'fcnn': ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


def train_and_eval(ModelClass, model_type, lr,
                   train_img, train_lm, train_y,
                   val_img, val_lm, val_y,
                   test_img, test_lm, test_y,
                   emotions, save_dir):
    criterion = nn.CrossEntropyLoss()
    if (save_dir / 'model.pth').exists():
        print(f'    [SKIP] model.pth exists, loading and evaluating...')
        model = ModelClass(num_classes=len(emotions)).to(device)
        test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)
        model.load_state_dict(torch.load(save_dir / 'model.pth', map_location=device, weights_only=True))
        r = full_evaluation(model, test_loader, criterion, device, model_type, emotions)
        return {
            'accuracy': float(r['test_accuracy']),
            'macro_f1': float(r['test_macro_f1']),
            'micro_f1': float(r['test_micro_f1']),
            'weighted_f1': float(r['test_weighted_f1']),
        }
    tr_loader = make_loader(train_img, train_lm, train_y, model_type, BATCH_SIZE)
    val_loader = make_loader(val_img, val_lm, val_y, model_type, BATCH_SIZE, False)
    test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)

    model = ModelClass(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / 'model.pth')
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, emotions)
    return {
        'accuracy': float(r['test_accuracy']),
        'macro_f1': float(r['test_macro_f1']),
        'micro_f1': float(r['test_micro_f1']),
        'weighted_f1': float(r['test_weighted_f1']),
    }


def late_fusion_eval(train_img, train_lm, train_y,
                     val_img, val_lm, val_y,
                     test_img, test_lm, test_y,
                     num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    if (save_dir / 'cnn.pth').exists():
        print('    [SKIP] cnn.pth exists')
    else:
        tr_cnn = make_loader(train_img, train_lm, train_y, 'cnn', BATCH_SIZE)
        val_cnn = make_loader(val_img, val_lm, val_y, 'cnn', BATCH_SIZE, False)
        opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                    device, 'cnn', EPOCHS, PATIENCE, str(save_dir / 'cnn.pth'))

    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    if (save_dir / 'fcnn.pth').exists():
        print('    [SKIP] fcnn.pth exists')
    else:
        tr_fcnn = make_loader(train_img, train_lm, train_y, 'fcnn', BATCH_SIZE)
        val_fcnn = make_loader(val_img, val_lm, val_y, 'fcnn', BATCH_SIZE, False)
        opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
        sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                    device, 'fcnn', EPOCHS, PATIENCE, str(save_dir / 'fcnn.pth'))

    cnn.load_state_dict(torch.load(save_dir / 'cnn.pth', map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / 'fcnn.pth', map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    def batched_softmax(model, arr, is_img, bs=32):
        outs = []
        with torch.no_grad():
            for i in range(0, len(arr), bs):
                chunk = arr[i:i+bs]
                if is_img:
                    t = torch.from_numpy(chunk).permute(0, 3, 1, 2).to(device)
                else:
                    t = torch.from_numpy(chunk).to(device)
                p = torch.softmax(model(t), dim=1).cpu().numpy()
                outs.append(p)
                del t
                torch.cuda.empty_cache()
        return np.concatenate(outs, axis=0)

    vc = batched_softmax(cnn, val_img, True)
    vf = batched_softmax(fcnn, val_lm, False)

    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w * vc + (1-w) * vf).argmax(axis=1)
        f = f1_score(val_y, preds, average='macro', zero_division=0)
        if f > best_wf1: best_wf1 = f; best_w = w
    print(f'    Best late-fusion weight (by val macro F1): cnn={best_w:.2f}')

    cnn_probs = batched_softmax(cnn, test_img, True)
    fcnn_probs = batched_softmax(fcnn, test_lm, False)
    preds = (best_w * cnn_probs + (1-best_w) * fcnn_probs).argmax(axis=1)
    m = metrics_triple(test_y, preds)
    m['best_cnn_weight'] = float(best_w)
    return m


def run_benchmark(num_classes, emotions, data_dir):
    print(f"\n{'='*70}")
    print(f'  RAF-DB {num_classes}-class (Official train/test split, B1)')
    print(f"{'='*70}")

    X_tr = np.load(data_dir / 'X_train_images.npy')
    L_tr = np.load(data_dir / 'X_train_landmarks.npy')
    y_tr = np.load(data_dir / 'y_train.npy')
    X_te = np.load(data_dir / 'X_test_images.npy')
    L_te = np.load(data_dir / 'X_test_landmarks.npy')
    y_te = np.load(data_dir / 'y_test.npy')

    # 10% of train -> val (stratified)
    from sklearn.model_selection import train_test_split
    idx_tr, idx_va = train_test_split(np.arange(len(y_tr)), test_size=0.1, stratify=y_tr, random_state=42)
    print(f'  Train: {len(idx_tr)}  Val: {len(idx_va)}  Test: {len(y_te)}')

    all_results = {}
    models_to_run = MODELS + [('Late_Fusion', None, 'late', 0.0001)]

    for model_name, ModelClass, model_type, lr in models_to_run:
        key = f'{model_name}_B1'
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)

        tr_img, tr_lm, tr_y = X_tr[idx_tr], L_tr[idx_tr], y_tr[idx_tr]
        v_img, v_lm, v_y = X_tr[idx_va], L_tr[idx_va], y_tr[idx_va]

        if model_type == 'late':
            r = late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                  X_te, L_te, y_te, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                X_te, L_te, y_te, emotions, save_dir)
        all_results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")

    save_path = OUTPUT_DIR / f'rafdb_{num_classes}c_results.json'
    with open(save_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return all_results

print('Helper functions ready.')

Helper functions ready.


## Run RAF-DB Benchmark

In [3]:
BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']

res_7c = run_benchmark(7, EMOTIONS_7, BENCHMARK_DIR / 'rafdb_7class')
res_4c = run_benchmark(4, EMOTIONS_4, BENCHMARK_DIR / 'rafdb_4class')


  RAF-DB 7-class (Official train/test split, B1)


  Train: 10408  Val: 1157  Test: 2884

  >> CNN_B1 ...


    [SKIP] model.pth exists, loading and evaluating...


Test Loss: 0.6858
Test Accuracy: 0.8152
Test Macro F1:    0.7294
Test Micro F1:    0.8152
Test Weighted F1: 0.8128

Classification Report:
              precision    recall  f1-score   support

     neutral       0.73      0.82      0.77       620
       happy       0.92      0.93      0.93      1136
         sad       0.76      0.75      0.76       448
       angry       0.80      0.64      0.71       148
     fearful       0.83      0.52      0.64        67
   disgusted       0.54      0.46      0.50       150
   surprised       0.84      0.78      0.81       315

    accuracy                           0.82      2884
   macro avg       0.77      0.70      0.73      2884
weighted avg       0.81      0.82      0.81      2884

    Macro=0.7294  Micro=0.8152  Weighted=0.8128

  >> FCNN_B1 ...


    [SKIP] model.pth exists, loading and evaluating...
Test Loss: 0.8181
Test Accuracy: 0.7136
Test Macro F1:    0.5781
Test Micro F1:    0.7136
Test Weighted F1: 0.7031

Classification Report:
              precision    recall  f1-score   support

     neutral       0.63      0.75      0.68       620
       happy       0.83      0.89      0.86      1136
         sad       0.62      0.54      0.58       448
       angry       0.61      0.57      0.59       148
     fearful       0.73      0.24      0.36        67
   disgusted       0.39      0.23      0.29       150
   surprised       0.73      0.64      0.68       315

    accuracy                           0.71      2884
   macro avg       0.65      0.55      0.58      2884
weighted avg       0.71      0.71      0.70      2884

    Macro=0.5781  Micro=0.7136  Weighted=0.7031

  >> Intermediate_B1 ...


...

    48      0.7318     0.7082     0.6764    0.7442   0.7026   0.000025  (2.0s)


    49      0.7302     0.7142     0.6788    0.7485   0.7047   0.000025  (2.1s)


    50      0.7393     0.7043     0.6706    0.7476   0.7039   0.000025  (2.2s)

Best: epoch 47, val_acc=0.7476, val_f1=0.7114
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/4c/Late_Fusion_B1/fcnn.pth


    Best late-fusion weight (by val macro F1): cnn=0.55


    Macro=0.8194  Micro=0.8415  Weighted=0.8415

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/rafdb_4c_results.json


## Ringkasan RAF-DB (Semua Metrik)

In [4]:
for nc_label, res in [('7-class', res_7c), ('4-class', res_4c)]:
    print(f"\n{'='*78}")
    print(f'  RAF-DB {nc_label} - sorted by Macro F1')
    print(f"{'='*78}")
    print(f"  {'Model':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for key in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[key]
        print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")


  RAF-DB 7-class - sorted by Macro F1
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  Intermediate_TL_B1         0.7440     0.8329     0.8322     0.8329
  CNN_TL_B1                  0.7407     0.8304     0.8265     0.8304
  CNN_B1                     0.7294     0.8152     0.8128     0.8152
  Late_Fusion_B1             0.7191     0.8093     0.8046     0.8093
  Intermediate_B1            0.6958     0.7854     0.7827     0.7854
  FCNN_B1                    0.5781     0.7136     0.7031     0.7136

  RAF-DB 4-class - sorted by Macro F1
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  Intermediate_TL_B1         0.8356     0.8533     0.8546     0.8533
  CNN_TL_B1                  0.8269     0.8447     0.8456     0.8447
  Late_Fusion_B1             0.8194     0.8415     0.8415     0.8415
  CNN_B1         